In [ ]:
!hf download mistralai/Mistral-7B-v0.1   --local-dir ~/mistral-7b-base

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_path = "/root/mistral-7b-base"
adapter_path = "/root/mistral-qlora-sentiment-adapter"
output_path = "/root/mistral-7b-merged"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

print("Loading base model in FP16...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("Loading adapter...")
model = PeftModel.from_pretrained(model, adapter_path)

print("Merging adapter into base model...")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

print("Done!")

In [ ]:
import os
for f in sorted(os.listdir("/root/mistral-7b-merged")):
    size = os.path.getsize(f"/root/mistral-7b-merged/{f}")
 print(f"{f:50s} {size/1e9:.2f} GB" if size > 1e8 else f"{f:50s} {size/1e3:.1f} KB")

config.json                                        0.7 KB
generation_config.json                             0.1 KB
model.safetensors                                  14.48 GB
tokenizer.json                                     3505.6 KB
tokenizer_config.json                              0.5 KB


In [ ]:
import subprocess
result = subprocess.run(
 "cd ~ && git clone https://github.com/ggerganov/llama.cpp && "
 "cd llama.cpp && pip install -r requirements.txt",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
print(result.stderr[-1000:] if result.stderr else "")

inux_2_17_x86_64.manylinux_2_28_x86_64.whl (221 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/221.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 14.0 MB/s eta 0:00:00
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9685 sha256=ed33bf71960f222945e96687ec120e2653e884eef9a010e64989042cf1f9b454
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget
  Attempting uninstall: requests
    Found existing installation: requests 2.33.0
    Uninstalling requests-2.33.0:
      Successfully uninstalled requests-2.33.0
  Attempting uninstall: pytest
    Found existing installation: pytest 9.0.3
    Uninstalling pytest-9.0.3:
      Successfully uninstalled pytest-9.0.3
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
  Attempting uninstall: prometheus-client
  

In [ ]:
result = subprocess.run(
 "cd ~/llama.cpp && python3 convert_hf_to_gguf.py "
 "/root/mistral-7b-merged "
 "--outfile /root/mistral-7b-merged-f16.gguf "
 "--outtype f16",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
print(result.stderr[-1000:] if result.stderr else "")

2Gbyte/s]
Writing:  81%|████████▏ | 11.8G/14.5G [00:08<00:01, 1.47Gbyte/s]
Writing:  83%|████████▎ | 12.0G/14.5G [00:08<00:01, 1.54Gbyte/s]
Writing:  84%|████████▍ | 12.2G/14.5G [00:09<00:01, 1.39Gbyte/s]
Writing:  86%|████████▌ | 12.4G/14.5G [00:09<00:01, 1.41Gbyte/s]
Writing:  87%|████████▋ | 12.7G/14.5G [00:09<00:01, 1.21Gbyte/s]
Writing:  89%|████████▉ | 12.9G/14.5G [00:09<00:01, 1.26Gbyte/s]
Writing:  90%|█████████ | 13.1G/14.5G [00:09<00:01, 1.14Gbyte/s]
Writing:  92%|█████████▏| 13.3G/14.5G [00:10<00:00, 1.24Gbyte/s]
Writing:  93%|█████████▎| 13.5G/14.5G [00:10<00:00, 1.17Gbyte/s]
Writing:  95%|█████████▍| 13.7G/14.5G [00:10<00:00, 1.30Gbyte/s]
Writing:  96%|█████████▋| 14.0G/14.5G [00:10<00:00, 1.29Gbyte/s]
Writing:  98%|█████████▊| 14.2G/14.5G [00:10<00:00, 1.33Gbyte/s]
Writing:  99%|█████████▉| 14.4G/14.5G [00:10<00:00, 1.29Gbyte/s]
Writing: 100%|██████████| 14.5G/14.5G [00:11<00:00, 1.31Gbyte/s]
INFO:hf-to-gguf:Model successfully exported to /root/mistral-7b-merged-f16.gguf


In [ ]:
result = subprocess.run(
 "cd ~/llama.cpp && cmake -B build && cmake --build build --config Release -j$(nproc)",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print(result.stderr[-1000:] if result.stderr else "")

lts
[ 93%] Built target llama-results
[ 94%] Building CXX object tools/server/CMakeFiles/server-context.dir/server-tools.cpp.o
[ 95%] Linking CXX executable ../bin/test-quant-type-selection
[ 95%] Built target test-quant-type-selection
[ 95%] Building CXX object tests/CMakeFiles/test-peg-parser.dir/peg-parser/test-python-dict-parser.cpp.o
[ 95%] Linking CXX executable ../bin/export-graph-ops
[ 96%] Building CXX object tests/CMakeFiles/test-backend-ops.dir/get-model.cpp.o
[ 96%] Built target export-graph-ops
[ 96%] Building CXX object tests/CMakeFiles/test-peg-parser.dir/peg-parser/test-unicode.cpp.o
[ 96%] Linking CXX executable ../bin/test-backend-ops
[ 96%] Built target test-backend-ops
[ 97%] Building CXX object tests/CMakeFiles/test-peg-parser.dir/get-model.cpp.o
[ 97%] Linking CXX executable ../../bin/llama-bench
[ 97%] Built target llama-bench
[ 98%] Linking CXX executable ../../bin/llama-tts
[ 98%] Built target llama-tts
[ 98%] Linking CXX executable ../bin/test-peg-parser
[ 98%

In [ ]:
result = subprocess.run(
 "cd ~/llama.cpp && ./build/bin/llama-quantize "
 "/root/mistral-7b-merged-f16.gguf "
 "/root/mistral-7b-merged-q4km.gguf "
 "Q4_K_M",
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
print(result.stderr[-1000:] if result.stderr else "")

main: quantize time = 75749.15 ms
main:    total time = 75749.15 ms

        - [  4096,   4096,      1,      1], type =    f16, converting to q4_K .. size =    32.00 MiB ->     9.00 MiB
[ 287/ 291] blk.31.attn_v.weight                 - [  4096,   1024,      1,      1], type =    f16, converting to q6_K .. size =     8.00 MiB ->     3.28 MiB
[ 288/ 291] blk.31.ffn_down.weight               - [ 14336,   4096,      1,      1], type =    f16, converting to q6_K .. size =   112.00 MiB ->    45.94 MiB
[ 289/ 291] blk.31.ffn_gate.weight               - [  4096,  14336,      1,      1], type =    f16, converting to q4_K .. size =   112.00 MiB ->    31.50 MiB
[ 290/ 291] blk.31.ffn_norm.weight               - [  4096,      1,      1,      1], type =    f32, size =    0.016 MiB
[ 291/ 291] blk.31.ffn_up.weight                 - [  4096,  14336,      1,      1], type =    f16, converting to q4_K .. size =   112.00 MiB ->    31.50 MiB
llama_model_quantize_impl: model size  = 13813.02 MiB (16.00 B

In [ ]:
result = subprocess.run(
 "~/llama.cpp/build/bin/llama-gguf /root/mistral-7b-merged-q4km.gguf r",
    shell=True, capture_output=True, text=True
)
print(result.stdout[:5000])
print(result.stderr[:1000] if result.stderr else "")

gguf_ex_read_0: version:      3
gguf_ex_read_0: alignment:   32
gguf_ex_read_0: data offset: 735296
gguf_ex_read_0: n_kv: 31
gguf_ex_read_0: kv[0]: key = general.architecture
gguf_ex_read_0: kv[1]: key = general.type
gguf_ex_read_0: kv[2]: key = general.name
gguf_ex_read_0: kv[3]: key = general.finetune
gguf_ex_read_0: kv[4]: key = general.basename
gguf_ex_read_0: kv[5]: key = general.size_label
gguf_ex_read_0: kv[6]: key = llama.block_count
gguf_ex_read_0: kv[7]: key = llama.context_length
gguf_ex_read_0: kv[8]: key = llama.embedding_length
gguf_ex_read_0: kv[9]: key = llama.feed_forward_length
gguf_ex_read_0: kv[10]: key = llama.attention.head_count
gguf_ex_read_0: kv[11]: key = llama.attention.head_count_kv
gguf_ex_read_0: kv[12]: key = llama.rope.freq_base
gguf_ex_read_0: kv[13]: key = llama.attention.layer_norm_rms_epsilon
gguf_ex_read_0: kv[14]: key = llama.attention.key_length
gguf_ex_read_0: kv[15]: key = llama.attention.value_length
gguf_ex_read_0: kv[16]: key = llama.vocab_si

In [ ]:
!/venv/main/bin/pip install gguf

from gguf import GGUFReader

reader = GGUFReader("/root/mistral-7b-merged-q4km.gguf")

for tensor in reader.tensors:
 print(f"{tensor.name:50s} {tensor.tensor_type.name}")

  Using cached gguf-0.18.0-py3-none-any.whl.metadata (4.5 kB)
Using cached gguf-0.18.0-py3-none-any.whl (114 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [gguf]3/4 [gguf]
output.weight                                      Q6_K
output_norm.weight                                 F32
token_embd.weight                                  Q4_K
blk.0.attn_k.weight                                Q4_K
blk.0.attn_norm.weight                             F32
blk.0.attn_output.weight                           Q4_K
blk.0.attn_q.weight                                Q4_K
blk.0.attn_v.weight                                Q6_K
blk.0.ffn_down.weight                              Q6_K
blk.0.ffn_gate.weight                              Q4_K
blk.0.ffn_norm.weight                              F32
blk.0.ffn_up.weight                                Q4_K
blk.1.attn_k.weight                                Q4_K
blk.1.attn_norm.weight                             F32
blk.1.attn_output.weight                  

In [ ]:
import sys
print(sys.executable)

/venv/main/bin/python
